# Spacetime from causal relationships — a computational experiment

**Question:** Can some geometric information (especially elapsed *proper time*) be recovered from just the relationships among discrete events?

This notebook implements a minimal example from **causal-set theory** in **1+1-dimensional flat spacetime**. It is a **numerical demonstration of a known reconstruction method**, *not evidence that physical spacetime is discrete*. We begin with ordinary Minkowski spacetime as a benchmark, generate discrete events in it, then reconstruct a small-case chain from its causal ordering. Repeated sweeps use the explicitly coordinate-aware fast algorithm. Absolute time also requires the externally calibrated sprinkling density.

The distinction matters: reconstructing geometry from relations is **not** the same as discovering a dynamical microscopic theory that *creates* spacetime or derives Einstein's equations.

## The experiment

For two timelike-separated events $A=(t,x)=(0,0)$ and $B=(T,0)$, use units $c=1$ so the true elapsed proper time is $\tau=T$.

Introduce *null coordinates* $u=t+x$, $v=t-x$. Points between $A$ and $B$ satisfy $0<u<T$, $0<v<T$; the spacetime volume of this region is $T^2/2$.

Randomly sprinkle events with density $\rho$ per unit spacetime volume. We use the strict timelike relation: **both** $u$ and $v$ increase. Null-related ties are excluded; they have probability zero in the continuous sprinkling law. We will compute the **longest chain** of such events, using only the ordering. At high density in this particular 1+1-dimensional model,

$$L \sim 2\sqrt{N}\approx \sqrt{2\rho}\,T, \qquad \widehat\tau = \frac{L}{\sqrt{2\rho}}.$$

Finite simulations have statistical scatter and systematic finite-density bias; convergence is the thing to test, not something to assume.

The Jacobian is $dt\,dx=du\,dv/2$, so $N\sim\mathrm{Poisson}(\rho T^2/2)$. Conditional on $N$, sorting by $u$ makes the ranks of the independent $v$ coordinates a uniform random permutation. Its longest increasing subsequence satisfies $L/\sqrt{N}\to2$. These facts give the normalization above. The boundary endpoints are not included in $L$.

Sources: [Brightwell & Gregory, *Structure of random discrete spacetime* (1991)](https://doi.org/10.1103/PhysRevLett.66.260); [Baik, Deift & Johansson, *On the distribution of the length of the longest increasing subsequence of random permutations* (1999)](https://arxiv.org/abs/math/9810105).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from causal_lab import (
    sprinkle, causal_order, longest_chain_order, longest_chain_fast,
    inferred_duration,
)

rng = np.random.default_rng(20261025)


def estimate_time(T, rho, rng):
    """Coordinate-aware fast sweep, using the tested package implementation."""
    diamond = sprinkle(T, rho, rng)
    return inferred_duration(len(longest_chain_fast(diamond)), rho)


## Experiment 1 — draw a causal set

Dots are events. The coloured connected path is a *longest causal chain*: each event on the chain could influence the next. Only the order relations are needed to find this chain; the positions are shown so we can interpret the result.

In [ ]:
T, rho = 1.0, 180.0
diamond = sprinkle(T, rho, rng)
u, v, t, x = diamond.u, diamond.v, diamond.t, diamond.x
# Convert to the causal order; then reconstruct a chain using ONLY that matrix.
order = causal_order(diamond)
chain = longest_chain_order(order)
assert len(chain) == len(longest_chain_fast(diamond))  # checks the fast equivalent algorithm

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(x, t, s=19, alpha=0.46, label='Sprinkled events')
x_chain = [0.0] + list(x[chain]) + [0.0]
t_chain = [0.0] + list(t[chain]) + [T]
ax.plot(x_chain, t_chain, linewidth=2.2, marker='o', markersize=4, label='Longest causal chain')
ax.plot([-T/2, 0, T/2, 0, -T/2], [T/2, 0, T/2, T, T/2], '--', alpha=0.45)
ax.scatter([0, 0], [0, T], s=55, marker='*', label='Endpoints A and B')
ax.set(xlabel='Space x (c=1)', ylabel='Time t', title='A causal interval in flat spacetime')
ax.set_aspect('equal', 'box')
ax.grid(alpha=0.15)
ax.legend(loc='upper right')
plt.show()
print(f'Sampled events: {len(u)}')
print(f'Longest interior chain: {len(chain)} events')
print(f'Estimated proper time: {len(chain)/np.sqrt(2*rho):.3f}  |  true proper time: {T:.3f}')


## Experiment 2 — recover different elapsed times

Repeat the random experiment several times for several intervals, keeping density fixed. The correct result would lie on the diagonal. Finite-density effects tend to produce an underestimate; watch how the approximation behaves rather than treating the formula as exact.
**Implementation note:** Experiment 1 really does reconstruct its chain from a matrix of relationships with coordinates discarded. In Experiments 2–4 we use an equivalent longest-increasing-subsequence algorithm on the generating null coordinates because it is much faster for thousands of events. These sweeps use the known embedding and do not demonstrate coordinate-independent reconstruction. The small-case agreement check only verifies the chain calculation.


In [ ]:
rho = 700.0
true_times = np.array([0.5, 1.0, 1.5, 2.0, 3.0])
repeats = 80
means, standard_errors = [], []
for T in true_times:
    readings = np.array([estimate_time(T, rho, rng) for _ in range(repeats)])
    means.append(np.mean(readings))
    standard_errors.append(np.std(readings, ddof=1) / np.sqrt(repeats))

fig, ax = plt.subplots(figsize=(7, 4.8))
ax.errorbar(true_times, means, yerr=standard_errors, fmt='o-', capsize=3,
            label='Reconstructed from causal chains')
ax.plot([0, 3.1], [0, 3.1], '--', alpha=.7, label='Perfect recovery')
ax.set(xlabel='True proper time', ylabel='Estimated proper time',
       title='Can elapsed time be recovered from causal order?')
ax.set_xlim(0, 3.1)
ax.set_ylim(0, 3.1)
ax.legend()
ax.grid(alpha=.2)
plt.show()
for T, measured in zip(true_times, means):
    print(f'True {T:.1f}: reconstructed {measured:.3f} ({measured/T:.1%} of truth)')


## Experiment 3 — improve the microscopic resolution

Here the true interval is always $T=1$. Increasing $\rho$ increases the average number of sprinkled events. The high-density estimate should approach the true elapsed time. Repeat with other random seeds and inspect both bias and variance.

In [ ]:
T = 1.0
rhos = np.array([30, 100, 300, 1_000, 4_000], dtype=float)
repeats = 100
means, sds = [], []
for rho in rhos:
    estimates = np.array([estimate_time(T, rho, rng) for _ in range(repeats)])
    means.append(estimates.mean())
    sds.append(estimates.std(ddof=1))

fig, ax = plt.subplots(figsize=(7, 4.8))
ax.errorbar(rhos, means, yerr=sds, fmt='o-', capsize=3,
            label='Mean ± 1 standard deviation')
ax.axhline(T, linestyle='--', label='True proper time')
ax.set(xscale='log', xlabel='Sprinkling density (events per spacetime volume)',
       ylabel='Inferred proper time', title='Continuum limit: improving reconstruction')
ax.legend()
ax.grid(alpha=.2)
plt.show()
for rho, est in zip(rhos, means):
    print(f'rho={int(rho):5}: inferred={est:.3f},  error={100*(est/T-1):+.1f}%')


## Experiment 4 — falsification: not every discrete order resembles spacetime

For roughly the same number of events, a **total order** (every event is causally comparable to every other event) has longest chain length equal to the number of events, whereas the 1+1-dimensional sprinkled order grows roughly as its square root. These are qualitatively different geometric signatures.

A model consisting of "dots and arrows" is **not** automatically a model of physical spacetime. It must generate something suitably like Lorentzian geometry, with the right effective dimension, dynamics and symmetries.

In [ ]:
rho, T = 600.0, 1.0
diamond = sprinkle(T, rho, rng)
n = diamond.count
causal_chain = len(longest_chain_fast(diamond))
total_order_chain = n
print(f'Events: {n}')
print(f'Sprinkled 1+1D causal chain length: {causal_chain}')
print(f'Total-order chain length:          {total_order_chain}')
print('These orders have different chain scaling; one statistic does not establish geometry.')


## What have we learned—and what have we **not** proved?

**Demonstrated:** In a causal set *known to come from* 1+1D flat spacetime, elapsed proper time can be approximately reconstructed using causal ordering and event density. Different order structures encode different geometric information.

**Not demonstrated:** That the actual universe is discrete; that Minkowski geometry emerges dynamically from a background-free microscopic system; that Einstein's equations follow; or that black-hole singularities disappear. Those are much harder problems.

### Your next genuine research questions

1. **Dimension:** Can you determine whether a network resembles 1+1D, 2+1D or 3+1D spacetime *without reading its hidden coordinates*? Explore the Myrheim–Meyer dimension estimator.
2. **Dynamics:** Can you define a local growth rule for events/causal links that produces stable continuum-like structure without embedding the model in a pre-existing spacetime?
3. **Curvature:** Can you reconstruct elapsed time for events sprinkled into *curved* spacetime, and find when reconstruction breaks down?
4. **Black holes:** Can a discrete model represent horizon-like causal structure while avoiding a singularity, without contradicting known exterior gravity?
5. **Evidence:** Identify a novel measurable prediction. Without this, a model is a thought experiment or mathematical construction, not a validated theory of nature.

### Follow existing research

- [Sorkin — Geometry from order: causal sets](https://www.einstein-online.info/en/spotlight/causal_sets/)
- [Causal sets and an emerging continuum (2024)](https://link.springer.com/article/10.1007/s10714-024-03281-1)
- [Jacobson — Thermodynamics of Spacetime (1995)](https://doi.org/10.1103/PhysRevLett.75.1260)
- [GWOSC — hands-on gravitational-wave Python tutorials](https://gwosc.org/tutorials/)

### To run

```bash
python -m pip install -e '.[dev]'
jupyter lab
```

Run these commands from the repository root. Then open this notebook from Jupyter and select **Run All**. The random generator has a fixed seed for reproducibility. Change the density, sample length and random seed to explore what survives.
